# HW04: ML and DL

Remember that these homework work as a completion grade. **You can skip one section without losing credit.**

## Load and Pre-process Text
We do sentiment analysis on the [Movie Review Data](https://www.cs.cornell.edu/people/pabo/movie-review-data/). If you would like to know more about the data, have a look at [the paper](https://www.cs.cornell.edu/home/llee/papers/pang-lee-stars.pdf) (but no need to do so).

In [2]:
# In this tutorial, we do sentiment analysis
# download the data
!curl -O https://www.cs.cornell.edu/people/pabo/movie-review-data/scale_data.tar.gz
!curl -O https://www.cs.cornell.edu/people/pabo/movie-review-data/scale_whole_review.tar.gz
 
!tar xf scale_data.tar.gz 
!tar xf scale_whole_review.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 3935k  100 3935k    0     0  2845k      0  0:00:01  0:00:01 --:--:-- 2857k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 8645k  100 8645k    0     0  6526k      0  0:00:01  0:00:01 --:--:-- 6539k


First, we have to load the data for which we provide the function below. Note how we also preprocess the text using gensim's simple_preprocess() function and how we already split the data into a train and test split.

In [3]:
import os
from gensim.utils import simple_preprocess
def load_data():
    examples, labels = [], []
    authors = os.listdir("scale_whole_review")
    for author in authors:
        path = os.listdir(os.path.join("scale_whole_review", author, "txt.parag"))
        fn_ids = os.path.join("scaledata", author, "id." + author)
        fn_ratings = os.path.join("scaledata", author, "rating." + author)
        with open(fn_ids) as ids, open(fn_ratings) as ratings:
            for idx, rating in zip(ids, ratings):
                labels.append(float(rating.strip()))
                filename_text = os.path.join("scale_whole_review", author, "txt.parag", idx.strip() + ".txt")
                with open(filename_text, encoding='latin-1') as f:
                    examples.append(" ".join(simple_preprocess(f.read())))
    return examples, labels
                  
X,y  = load_data()
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
print ("text:", X_train[0], "\nlabel:", y_train[0])

text: bloody child the director writer cinematographer nina menkes screenwriter tinka menkes editors nina and tina menkes cast tinka menkes captain sherry sibley murdered wife robert mueller murderer russ little sergeant jack hara enlisted man runtime mirage reviewed by dennis schwartz an amazingly strange film confusing and not thoroughly enjoyable but film found more interesting than thought possible at first viewing this experimental film in minimalist story telling film consisting of disturbing visualizations and almost no dialogue had concept that was greater than how the film turned out it felt at times like was watching paint dry on the wall but the reward for sitting through those excruciatingly redundant scenes was in seeing something different something that cast spell of sorcery over terrible incident as believe the film in its unique and sometimes shrill voice does justice in commenting on the violence in american society especially against women the film uses its impressio

## Vectorize the data

In [6]:
# train a TF_IDF Vectorizer on X_train and vectorize X_train and X_test
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(min_df=0.01, # at min 1% of docs
                        max_df=.5,  
                        stop_words='english',
                        ngram_range=(1,2))

vec.fit(X_train)

X_train_tfidf = vec.transform(X_train)
X_test_tfidf = vec.transform(X_test)

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler(with_mean=False)

scaler.fit(X_train_tfidf)

X_train_scaled = scaler.transform(X_train_tfidf)
X_test_scaled = scaler.transform(X_test_tfidf)

## ElasticNet

In [8]:
from sklearn.linear_model import ElasticNet

en = ElasticNet(alpha=0.01)
en.fit(X_train_scaled, y_train)

y_pred = en.predict(X_test_scaled)

from sklearn.metrics import r2_score, mean_squared_error
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MSE: 0.016535782264671804
R2: 0.4977289048964286


## Logistic Regression

Next, we train an OLS model doing binary prediction on these movie reviews. Two get two bins, we transform the continuous ratings into two classes, where one class contains all the negative ratings (value < 0.5), the other class all the positive ratings (value > 0.5)

In [10]:
y_train = [1 if i >= 0.5 else 0 for i in y_train]
y_test = [1 if i >= 0.5 else 0 for i in y_test]


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

def map_predictions(predicted):
    return [1 if i > 0.5 else 0 for i in predicted]

logistic_regression = LogisticRegression()
logistic_regression.fit(X_train_scaled, y_train)

y_prob = logistic_regression.predict_proba(X_test_scaled)[:, 1]
y_pred = map_predictions(y_prob)

print("Accuracy:", accuracy_score(y_test, y_pred))

feature_names = np.array(vec.get_feature_names_out())
top10_idx = np.argsort(logistic_regression.coef_[0])[-10:][::-1]
print("\n10 most informative words:")
for word, coef in zip(feature_names[top10_idx], logistic_regression.coef_[0][top10_idx]):
    print(f"  {word}: {coef:.4f}")

Accuracy: 0.8111380145278451

10 most informative words:
  great: 0.2331
  surprisingly: 0.2182
  best: 0.2119
  effective: 0.2049
  fascinating: 0.1984
  success: 0.1942
  punches: 0.1941
  elements: 0.1906
  brilliant: 0.1902
  fine: 0.1901


# Deep Learning

## MLP

In [13]:
#Import the AG news dataset (same as hw01)
#Download them from here 
#!wget https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv

import pandas as pd
import nltk
df = pd.read_csv('train.csv')

df.columns = ["label", "title", "lead"]
label_map = {1:"world", 2:"sport", 3:"business", 4:"sci/tech"}
def replace_label(x):
	return label_map[x]
df["label"] = df["label"].apply(replace_label) 
df["text"] = df["title"] + " " + df["lead"]
df = df.sample(n=10000) # # only use 10K datapoints
df.head()

,label,title,lead,text
116764,sci/tech,Phantom of the Xbox,After being dropped for release outside of Jap...,Phantom of the Xbox After being dropped for re...
64588,sport,Mutu #39;confessed #39; to taking drugs,"Becali, the most senior of Mutu #39;s team of ...",Mutu #39;confessed #39; to taking drugs Becal...
18931,world,Signs of Discord Emerge as Indo-Pak Ministers ...,NEW DELHI (Reuters) - Signs of discord emerge...,Signs of Discord Emerge as Indo-Pak Ministers ...
10661,business,"TD Buys US Retail Bank, Ups Dividend","Toronto Dominion Bank (TD.TO: Quote, Profile, ...","TD Buys US Retail Bank, Ups Dividend Toronto D..."
92829,sci/tech,Sony Introduces New Mac-Compatible Double Laye...,"SAN JOSE, Calif., Nov. 16 /PRNewswire-FirstCal...",Sony Introduces New Mac-Compatible Double Laye...


In [14]:
# create a new variable "business" that takes value 1 if the label is business and 0 otherwise
df['business'] = df['label'].apply(lambda x: int(x=='business'))
y = df['business'].values
df['business'].head()

116764    0
64588     0
18931     0
10661     1
92829     0
Name: business, dtype: int64

In [15]:
import spacy
nlp = spacy.load('en_core_web_sm')
from sklearn.feature_extraction.text import CountVectorizer

def tokenize(x):
    return [w.lemma_.lower() for w in nlp(x) if not w.is_stop and not w.is_punct and not w.is_digit]
df["tokens"] = df["text"].apply(lambda x: tokenize(x))
df["preprocessed"] = df['tokens'].apply(lambda x: ' '.join(x))

cv = CountVectorizer(min_df=0.01, max_df=0.5)
X = cv.fit_transform(df["preprocessed"])
print("Vocabulary size:", len(cv.vocabulary_))
print("Feature matrix shape:", X.shape)

Vocabulary size: 390
Feature matrix shape: (10000, 390)


Your goal here is to use features from the Vectorized text to predict whether the snippet is from a business article.

In [18]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torchsummary import summary
import numpy as np
import copy

# splits
X_arr = X.toarray().astype(np.float32)
X_train, X_test, y_train, y_test = train_test_split(X_arr, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader   = DataLoader(TextDataset(X_val,   y_val),   batch_size=64)

# MLP
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 1),         nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

input_dim = X_train.shape[1]
model = MLP(input_dim)
summary(model, input_size=(input_dim,))

# training with early stopping
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()
patience, best_val_acc, no_improve, best_weights = 5, 0, 0, None

for epoch in range(100):
    model.train()
    for X_b, y_b in train_loader:
        optimizer.zero_grad()
        criterion(model(X_b), y_b).backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        preds = (model(torch.FloatTensor(X_val)) > 0.5).int().tolist()
    val_acc = accuracy_score(y_val.tolist(), preds)

    if val_acc > best_val_acc:
        best_val_acc, best_weights, no_improve = val_acc, copy.deepcopy(model.state_dict()), 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} (best val acc: {best_val_acc:.4f})")
            break

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | val_acc: {val_acc:.4f}")

# evaluate best model on test set
model.load_state_dict(best_weights)
model.eval()
with torch.no_grad():
    test_preds = (model(torch.FloatTensor(X_test)) > 0.5).int().tolist()
print(f"\nTest accuracy: {accuracy_score(y_test.tolist(), test_preds):.4f}")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                  [-1, 256]         100,096
              ReLU-2                  [-1, 256]               0
           Dropout-3                  [-1, 256]               0
            Linear-4                  [-1, 128]          32,896
              ReLU-5                  [-1, 128]               0
           Dropout-6                  [-1, 128]               0
            Linear-7                    [-1, 1]             129
           Sigmoid-8                    [-1, 1]               0
Total params: 133,121
Trainable params: 133,121
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.01
Params size (MB): 0.51
Estimated Total Size (MB): 0.52
----------------------------------------------------------------
Early stopping at epoch 7 (best val acc: 0

In [17]:
!pip install torchsummary -q